In [1]:
import os
import subprocess
import sys
USER = "nikhilrk1"
REPO = "recommender-svd"
if os.path.exists(REPO):
  subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
  subprocess.run(
  ["git", "clone", f"https://github.com/{USER}/{REPO}.git", REPO],
  check=True)

sys.path.insert(0, f"{REPO}/src")
from evaluate import split_ratings, to_matrix, rmse
from models import bias_baseline, baseline_matrix, rank_k



Imports functions from other milestones

In [2]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
if not os.path.exists("ml-100k"):
  urllib.request.urlretrieve(
    "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
    "ml-100k.zip")
  zipfile.ZipFile("ml-100k.zip").extractall()
ratings = pd.read_csv(
  "ml-100k/u.data", sep="\t",
  names=["user", "item", "rating", "time"])
train, test = split_ratings(ratings, seed=0)
fit, val = split_ratings(train, seed=1)
print("fit:", len(fit), "validation:", len(val), "test:", len(test))

titles = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
header=None, usecols=[1])[1].to_numpy()
print(len(titles), titles[49])

fit: 81200 validation: 8832 test: 9968
1682 Star Wars (1977)


loads ratings and titles

In [3]:
R = to_matrix(ratings)
known = ~np.isnan(R)
B = baseline_matrix(R)
X = np.where(known, R - B, 0.0)
U, s, Vt = np.linalg.svd(X, full_matrices=False)
n_ratings = known.sum(axis=0)
popular = n_ratings >= 50
print("popular movies:", popular.sum())

popular movies: 603


fit model 2 on the ratings

In [4]:
def show_factor(f, n=6):
  """Popular movies at the two ends of hidden factor f (f = 1, 2, ...)."""
  v = Vt[f - 1]
  idx = np.where(popular)[0]
  order = idx[np.argsort(v[idx])]
  print(f"Factor {f}, one end:")
  for m in order[:n]:
    print(f" {v[m]:+.3f} {titles[m]}")
  print(f"Factor {f}, other end:")
  for m in order[::-1][:n]:
    print(f" {v[m]:+.3f} {titles[m]}")
show_factor(1)

Factor 1, one end:
 -0.087 Clockwork Orange, A (1971)
 -0.079 Trainspotting (1996)
 -0.072 Leaving Las Vegas (1995)
 -0.072 Pulp Fiction (1994)
 -0.069 Brazil (1985)
 -0.067 Fargo (1996)
Factor 1, other end:
 +0.211 Independence Day (ID4) (1996)
 +0.175 Return of the Jedi (1983)
 +0.149 Raiders of the Lost Ark (1981)
 +0.149 Back to the Future (1985)
 +0.136 Braveheart (1995)
 +0.134 Forrest Gump (1994)


Lists the hidden factors for each movie

In [6]:
def find(word):
  """Print the ID and title of every movie whose title contains word."""
  for m, t in enumerate(titles):
    if word.lower() in t.lower():
      print(m + 1, t)

find("alien")

my_ratings = {50: 5, 172: 5, 183: 5, 195: 5, 89: 4, 1: 4,
143: 1, 451: 1, 88: 1, 739: 2}

me = np.full((1, R.shape[1]), np.nan)
for movie, stars in my_ratings.items():
  me[0, movie - 1] = stars
R_me = np.vstack([R, me])
print(R_me.shape)
known_me = ~np.isnan(R_me)
B_me = baseline_matrix(R_me)
X_me = np.where(known_me, R_me - B_me, 0.0)
U, s, Vt = np.linalg.svd(X_me, full_matrices=False)
P = rank_k(U, s, Vt, 10) + B_me

mine = P[-1]
base = B_me[-1]
can_pick = popular & np.isnan(R_me[-1])
def top(scores, n=10):
  """The n highest-scoring movies among those we may recommend."""
  return np.argsort(-np.where(can_pick, scores, -np.inf))[:n]
print("Model 2 recommends:")
for m in top(mine):
  print(f" {mine[m]:.2f} {titles[m]}")
print("Baseline recommends:")
for m in top(base):
  print(f" {base[m]:.2f} {titles[m]}")
print("in both lists:", len(set(top(mine)) & set(top(base))))

personal = mine - base
print("Moved up most for me:")
for m in top(personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")
print("Moved down most for me:")
for m in top(-personal, 6):
  print(f" {personal[m]:+.2f} {titles[m]}")

176 Aliens (1986)
183 Alien (1979)
343 Alien: Resurrection (1997)
665 Alien 3 (1992)
(944, 1682)
Model 2 recommends:
 4.28 Close Shave, A (1995)
 4.26 Wrong Trousers, The (1993)
 4.20 Fargo (1996)
 4.15 Casablanca (1942)
 4.14 Raiders of the Lost Ark (1981)
 4.11 Wallace & Gromit: The Best of Aardman Animation (1996)
 4.08 Godfather, The (1972)
 4.07 Schindler's List (1993)
 4.07 Shawshank Redemption, The (1994)
 4.06 Return of the Jedi (1983)
Baseline recommends:
 4.23 Close Shave, A (1995)
 4.15 Wrong Trousers, The (1993)
 4.13 Schindler's List (1993)
 4.13 Shawshank Redemption, The (1994)
 4.09 Casablanca (1942)
 4.08 Wallace & Gromit: The Best of Aardman Animation (1996)
 4.08 Usual Suspects, The (1995)
 4.06 Good Will Hunting (1997)
 4.01 Titanic (1997)
 4.01 12 Angry Men (1957)
in both lists: 6
Moved up most for me:
 +0.34 Fargo (1996)
 +0.34 Return of the Jedi (1983)
 +0.25 Star Trek: The Wrath of Khan (1982)
 +0.22 Star Trek: First Contact (1996)
 +0.21 2001: A Space Odyssey (1

Testing what the model would recommend with a test reccommendation.